# SIC: OD-Wuhan

Default `replay` verifies saved results. Set `ARTICLE_MODE=train` before starting the kernel to refit.
The evidence publication flag defaults to false; `plot_results.ipynb` and its saved outputs are never edited or executed here.

## Read data

In [1]:
SHORT = 'SIC'
CASE = 'OD-Wuhan'

from pathlib import Path
import hashlib
import json
import os
import random
import shutil
import sys
import time
import platform
import numpy as np
import pandas as pd
import torch

SCRIPTS = next(p for p in [Path.cwd(), *Path.cwd().parents,
                           Path.cwd() / 'article/scripts', Path.cwd() / 'scripts']
               if (p / 'experiment_manifest.json').is_file())
SCRIPTS = SCRIPTS.resolve()
sys.path.insert(0, str(SCRIPTS))
from models.experiment_adapter import build_trainer
import models
assert Path(models.__file__).resolve().parent == SCRIPTS / 'models', 'Restart the kernel'
MODE = os.getenv('ARTICLE_MODE', 'replay')  # replay saved results; train refits all selected jobs
RUN_ID = os.getenv('ARTICLE_RUN_ID', 'replay' if MODE == 'replay' else 'retrain-v1')
PUBLISH_EVIDENCE = os.getenv('ARTICLE_PUBLISH_EVIDENCE', '0') == '1'
SEEDS = list(range(1, 11))
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.set_num_threads(4)
assert MODE in ('replay', 'train')
manifest = json.loads((SCRIPTS / 'experiment_manifest.json').read_text())
manifest_digest = hashlib.sha256((SCRIPTS / 'experiment_manifest.json').read_bytes()).hexdigest()
CASE_JOBS = [j for j in manifest['jobs'] if j['short'] == SHORT]
DATA_DIR = SCRIPTS / 'data' / SHORT
OUTPUT_DIR = SCRIPTS / 'outputs' / RUN_ID / SHORT
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
contract = json.loads((DATA_DIR / 'contract.json').read_text())
assert hashlib.sha256((DATA_DIR / 'ordinary.npz').read_bytes()).hexdigest() == contract['data_sha256']
data = np.load(DATA_DIR / 'ordinary.npz')
partitions = [data[f'{part}_ids'] for part in ('train', 'validation', 'test')]
assert len(np.unique(np.concatenate(partitions))) == len(data['x'])
if SHORT in ('SHP', 'GVI'):
    group_sets = [set(data['groups'][ids]) for ids in partitions]
    assert all(group_sets[i].isdisjoint(group_sets[j]) for i in range(3) for j in range(i))
case_metrics, case_predictions, case_bandwidths, case_groups, case_ols = [], [], [], [], []
pd.DataFrame([{'case': CASE, 'rows': len(data['x']), 'features': data['x'].shape[1],
               'coordinates': data['u'].shape[1], 'train': len(partitions[0]),
               'validation': len(partitions[1]), 'test': len(partitions[2]),
               'mode': MODE, 'jobs': len(CASE_JOBS)}])

,case,rows,features,coordinates,train,validation,test,mode,jobs
0,OD-Wuhan,13807,35,4,9618,2104,2085,replay,14


## Model experiments

### DLAN

In [3]:
model_jobs = [j for j in CASE_JOBS if j['model'] in ('dlan',)]
pd.DataFrame([{'job':j['id'], 'parameters':j['parameters'], 'candidates':j['candidates'],
               'roles':', '.join(j['roles'])} for j in model_jobs])

,job,parameters,candidates,roles
0,dlan_main_neighbors_factorial_w12_k512,7069,512,"main, neighbors, factorial"
1,dlan_archived_main_w10_k1024,6957,1024,archived_main
2,dlan_capacity_factorial_w12_k1024,7069,1024,"capacity, factorial"
3,dlan_neighbors_factorial_w12_k2048,7069,2048,"neighbors, factorial"
4,dlan_width16_factorial_w16_k1024,10773,1024,"width16, factorial"
5,dlan_factorial_w10_k512,5541,512,factorial
6,dlan_factorial_w10_k1024,5541,1024,factorial
7,dlan_factorial_w10_k2048,5541,2048,factorial
8,dlan_factorial_w16_k512,10773,512,factorial
9,dlan_factorial_w16_k2048,10773,2048,factorial


In [4]:
# Training is explicit. The default replay mode never enters this block.
if MODE == 'train':
    for job in model_jobs:
        d = np.load(DATA_DIR / job['data'])
        source = SCRIPTS / job['reference_dir']
        original = json.loads((source / 'configuration.json').read_text())
        target_inverse = original['contract'].get('target_inverse')
        directory = OUTPUT_DIR / job['id']
        directory.mkdir(exist_ok=True)
        configuration = {'specification': job, 'mode': MODE, 'manifest_digest': manifest_digest,
                         'training': {**job['training'], 'device': DEVICE},
                         'data_sha256': hashlib.sha256((DATA_DIR / job['data']).read_bytes()).hexdigest(),
                         'model_hashes': {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                                          for p in sorted((SCRIPTS / 'models').glob('*.py'))},
                         'python': platform.python_version(), 'torch': torch.__version__}
        digest = hashlib.sha256(json.dumps(configuration, sort_keys=True).encode()).hexdigest()
        configuration['config_digest'] = digest
        config_path = directory / 'configuration.json'
        if config_path.exists():
            assert json.loads(config_path.read_text())['config_digest'] == digest, 'Choose a new RUN_ID'
        else:
            config_path.write_text(json.dumps(configuration, indent=2))
        for seed in SEEDS:
            metric_path = directory / f'seed_{seed}_metrics.csv'
            if metric_path.exists():
                assert set(pd.read_csv(metric_path).config_digest) == {digest}
                assert all((directory / f'seed_{seed}_{s}').is_file()
                           for s in ('checkpoint.pt', 'predictions.npz', 'history.csv', 'audit.csv'))
                continue
            random.seed(seed)
            np.random.seed(seed)
            torch.manual_seed(seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(seed)
            trainer = build_trainer(job, DEVICE)
            started = time.perf_counter()
            history = trainer.fit(d['x'], d['u'], d['y'], d['train_ids'], d['validation_ids'], verbose=False)
            elapsed = time.perf_counter() - started
            archive, scores = {}, []
            for split in ('validation', 'test'):
                ids = d[f'{split}_ids']
                prediction = trainer.predict(d['x'][ids], d['u'][ids]).numpy().astype(float)
                if target_inverse:
                    prediction = np.maximum(np.expm1(prediction), 0)
                actual = d['actual'][ids].astype(float)
                assert np.isfinite(prediction).all()
                error = prediction - actual
                archive.update({f'{split}_row_id': d['row_id'][ids], f'{split}_actual': actual,
                                f'{split}_predicted': prediction, f'{split}_group_id': d['groups'][ids]})
                scores.append(dict(case=CASE, model=job['model'], seed=seed, split=split,
                    protocol=contract['split_policy'], scoring=contract['scoring'], config_digest=digest,
                    parameters=job['parameters'], epochs=len(history['train_loss']), seconds=elapsed,
                    rmse=float(np.sqrt(np.mean(error**2))), mae=float(np.mean(abs(error))),
                    r2=float(1 - np.sum(error**2) / np.sum((actual - actual.mean())**2))))
            state = dict(model_state=trainer.model.state_dict(), x_mean=trainer.x_scaler.mean,
                         x_std=trainer.x_scaler.std, y_mean=trainer.y_scaler.mean, y_std=trainer.y_scaler.std,
                         coord_center=trainer.coord_scaler.center, coord_scale=trainer.coord_scaler.scale,
                         context_ids=trainer.context_ids, context_x=trainer.context_x,
                         context_coordinates=trainer.context_coordinates, context_y=trainer.context_y)
            torch.save(dict(state=state, configuration=configuration, seed=seed),
                       directory / f'seed_{seed}_checkpoint.pt')
            np.savez_compressed(directory / f'seed_{seed}_predictions.npz', **archive)
            pd.DataFrame(history).to_csv(directory / f'seed_{seed}_history.csv', index=False)
            # Reload the checkpoint into a fresh model before declaring the seed complete.
            restored = build_trainer(job, DEVICE)
            saved = torch.load(directory / f'seed_{seed}_checkpoint.pt', map_location='cpu', weights_only=False)['state']
            restored.model.load_state_dict(saved['model_state'], strict=True)
            for name in ('context_ids', 'context_x', 'context_coordinates', 'context_y'):
                setattr(restored, name, saved[name])
            restored.x_scaler.mean, restored.x_scaler.std = saved['x_mean'], saved['x_std']
            restored.y_scaler.mean, restored.y_scaler.std = saved['y_mean'], saved['y_std']
            restored.coord_scaler.center, restored.coord_scaler.scale = saved['coord_center'], saved['coord_scale']
            audits = []
            for split in ('validation', 'test'):
                ids = d[f'{split}_ids']
                prediction = restored.predict(d['x'][ids], d['u'][ids]).numpy().astype(float)
                if target_inverse:
                    prediction = np.maximum(np.expm1(prediction), 0)
                np.testing.assert_allclose(prediction, archive[f'{split}_predicted'], rtol=1e-5, atol=1e-6)
                audits.append(dict(seed=seed, split=split, rows=len(ids), prediction_reload_pass=True))
            pd.DataFrame(audits).to_csv(directory / f'seed_{seed}_audit.csv', index=False)
            pd.DataFrame(scores).to_csv(metric_path, index=False)
            del trainer, restored, state, saved
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

In [5]:
model_records = []
for job in model_jobs:
    d = np.load(DATA_DIR / job['data'])
    source = SCRIPTS / job['reference_dir'] if MODE == 'replay' else OUTPUT_DIR / job['id']
    map_u = np.load(DATA_DIR / 'map_coordinates.npz')['coordinates']
    for seed in SEEDS:
        if MODE == 'replay':
            for suffix in ('metrics.csv', 'predictions.npz', 'checkpoint.pt', 'audit.csv', 'history.csv'):
                name = f'seed_{seed}_{suffix}'
                assert hashlib.sha256((source / name).read_bytes()).hexdigest() == job['files'][name]
        frame = pd.read_csv(source / f'seed_{seed}_metrics.csv')
        prediction = np.load(source / f'seed_{seed}_predictions.npz')
        audit = pd.read_csv(source / f'seed_{seed}_audit.csv')
        assert len(frame) == 2 and audit.prediction_reload_pass.all()
        checkpoint = torch.load(source / f'seed_{seed}_checkpoint.pt', map_location='cpu', weights_only=False)
        state = checkpoint['state']
        assert set(state['context_ids'].tolist()) == set(d['train_ids'].tolist())
        for split in ('validation', 'test'):
            ids = d[f'{split}_ids']
            np.testing.assert_array_equal(prediction[f'{split}_row_id'], d['row_id'][ids])
            actual = prediction[f'{split}_actual'].astype(float)
            predicted = prediction[f'{split}_predicted'].astype(float)
            np.testing.assert_allclose(actual, d['actual'][ids], rtol=1e-6, atol=1e-6)
            error = predicted - actual
            values = dict(rmse=np.sqrt(np.mean(error**2)), mae=np.mean(abs(error)),
                          r2=1 - np.sum(error**2) / np.sum((actual - actual.mean())**2))
            row = frame[frame.split == split].iloc[0].to_dict()
            for key, value in values.items():
                np.testing.assert_allclose(value, row[key], rtol=2e-5, atol=1e-6)
            row.update(values, case=CASE, model_label=job['label'], job_id=job['id'])
            if SHORT == 'SIC':
                tail = actual >= np.quantile(actual, .99)
                row['upper_1pct_sse_share'] = np.sum(error[tail]**2) / np.sum(error**2)
                row['body_rmse'] = np.sqrt(np.mean(error[~tail]**2))
                row['log1p_rmse'] = np.sqrt(np.mean((np.log1p(predicted)-np.log1p(actual))**2))
            if SHORT == 'SHP':
                details = pd.DataFrame(dict(group=d['groups'][ids], row_id=d['row_id'][ids],
                                            actual=actual, predicted=predicted, squared=error**2))
                row['equal_group_rmse'] = np.sqrt(details.groupby('group').squared.mean().mean())
                for group, g in details.groupby('group'):
                    index = np.flatnonzero(d['row_id'] == g.row_id.iloc[0])[0]
                    case_groups.append(dict(case=CASE, model=job['label'], job_id=job['id'], seed=seed,
                        split=split, group=group, n=len(g), rmse=np.sqrt(g.squared.mean()),
                        mean_residual=(g.predicted-g.actual).mean(), x=map_u[index,0], y=map_u[index,1]))
            case_metrics.append(row)
            model_records.append(row)
            if split == 'test':
                case_predictions.append(pd.DataFrame(dict(case=CASE, model=job['label'], job_id=job['id'],
                    seed=seed, row_id=d['row_id'][ids], actual=actual, predicted=predicted,
                    residual=error, group=d['groups'][ids], x=map_u[ids,0], y=map_u[ids,1])))
        if job['model'] == 'dlan':
            names = {'CE':['Space x','Space y','Time'], 'SIC':['Origin x','Origin y','Destination x','Destination y'],
                     'SHP':['Space x','Space y'], 'GVI':['Space x','Space y','Time']}[SHORT]
            for key, value in state['model_state'].items():
                if key.endswith('raw_bandwidth'):
                    bandwidth = (torch.nn.functional.softplus(value) + 1e-6).numpy()
                    for axis in range(bandwidth.shape[0]):
                        case_bandwidths.append(dict(case=CASE, job_id=job['id'], seed=seed,
                                                    axis=names[axis], bandwidth=float(bandwidth[axis,0])))
        if SHORT == 'GVI' and job['model'] == 'gcnnwr':
            ids = d['test_ids']
            sx = (torch.as_tensor(d['x'][ids], dtype=torch.float32)-state['x_mean']) / state['x_std']
            design = torch.cat((torch.ones_like(sx[:,:1]), sx), 1)
            ols = (design @ state['model_state']['network.ols_coefficients'] * state['y_std'] + state['y_mean']).numpy()
            actual = d['actual'][ids]
            case_ols.append(dict(case=CASE, seed=seed, rmse=np.sqrt(np.mean((ols-actual)**2))))
            if seed == 1:
                case_predictions.append(pd.DataFrame(dict(case=CASE, model='OLS buffer', job_id=job['id'],
                    seed=seed, row_id=d['row_id'][ids], actual=actual, predicted=ols,
                    residual=ols-actual, group=d['groups'][ids], x=map_u[ids,0], y=map_u[ids,1])))
pd.DataFrame(model_records).query("split == 'test'").groupby('job_id')[['rmse','mae','r2']].agg(['mean','std'])

rmse                    mae  \
                                              mean         std       mean   
job_id                                                                      
dlan_archived_main_w10_k1024            919.923195  424.903179  79.699137   
dlan_capacity_factorial_w12_k1024       608.731112  252.453147  52.083653   
dlan_factorial_w10_k1024                620.004445  165.681108  54.391805   
dlan_factorial_w10_k2048                591.824864  195.292671  53.214912   
dlan_factorial_w10_k512                 624.398136  154.943497  54.825070   
dlan_factorial_w16_k2048                665.206606  148.687082  55.554848   
dlan_factorial_w16_k512                 636.554337  118.344194  54.535738   
dlan_main_neighbors_factorial_w12_k512  565.276291  154.306900  50.281942   
dlan_neighbors_factorial_w12_k2048      587.598977  171.150757  51.074914   
dlan_width16_factorial_w16_k1024        609.928123  104.632089  52.938669   

                                                         r2            
                                              std      mean       std  
job_id                                                                 
dlan_archived_main_w10_k1024            30.630184  0.719298  0.270617  
dlan_capacity_factorial_w12_k1024       11.514114  0.880925  0.112195  
dlan_factorial_w10_k1024                 9.559800  0.886158  0.055329  
dlan_factorial_w10_k2048                10.580296  0.892983  0.063681  
dlan_factorial_w10_k512                  8.369638  0.885498  0.056731  
dlan_factorial_w16_k2048                 8.640161  0.871330  0.055746  
dlan_factorial_w16_k512                  6.742376  0.883738  0.044940  
dlan_main_neighbors_factorial_w12_k512   8.035142  0.905120  0.050597  
dlan_neighbors_factorial_w12_k2048       9.060719  0.896586  0.056766  
dlan_width16_factorial_w16_k1024         5.998548  0.893739  0.034249

### GeoAggregator

In [6]:
model_jobs = [j for j in CASE_JOBS if j['model'] in ('geoaggregator',)]
pd.DataFrame([{'job':j['id'], 'parameters':j['parameters'], 'candidates':j['candidates'],
               'roles':', '.join(j['roles'])} for j in model_jobs])

,job,parameters,candidates,roles
0,geoaggregator_main_capacity,7069,256,"main, capacity"
1,geoaggregator_archived_main,6957,256,archived_main


In [7]:
# Training is explicit. The default replay mode never enters this block.
if MODE == 'train':
    for job in model_jobs:
        d = np.load(DATA_DIR / job['data'])
        source = SCRIPTS / job['reference_dir']
        original = json.loads((source / 'configuration.json').read_text())
        target_inverse = original['contract'].get('target_inverse')
        directory = OUTPUT_DIR / job['id']
        directory.mkdir(exist_ok=True)
        configuration = {'specification': job, 'mode': MODE, 'manifest_digest': manifest_digest,
                         'training': {**job['training'], 'device': DEVICE},
                         'data_sha256': hashlib.sha256((DATA_DIR / job['data']).read_bytes()).hexdigest(),
                         'model_hashes': {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                                          for p in sorted((SCRIPTS / 'models').glob('*.py'))},
                         'python': platform.python_version(), 'torch': torch.__version__}
        digest = hashlib.sha256(json.dumps(configuration, sort_keys=True).encode()).hexdigest()
        configuration['config_digest'] = digest
        config_path = directory / 'configuration.json'
        if config_path.exists():
            assert json.loads(config_path.read_text())['config_digest'] == digest, 'Choose a new RUN_ID'
        else:
            config_path.write_text(json.dumps(configuration, indent=2))
        for seed in SEEDS:
            metric_path = directory / f'seed_{seed}_metrics.csv'
            if metric_path.exists():
                assert set(pd.read_csv(metric_path).config_digest) == {digest}
                assert all((directory / f'seed_{seed}_{s}').is_file()
                           for s in ('checkpoint.pt', 'predictions.npz', 'history.csv', 'audit.csv'))
                continue
            random.seed(seed)
            np.random.seed(seed)
            torch.manual_seed(seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(seed)
            trainer = build_trainer(job, DEVICE)
            started = time.perf_counter()
            history = trainer.fit(d['x'], d['u'], d['y'], d['train_ids'], d['validation_ids'], verbose=False)
            elapsed = time.perf_counter() - started
            archive, scores = {}, []
            for split in ('validation', 'test'):
                ids = d[f'{split}_ids']
                prediction = trainer.predict(d['x'][ids], d['u'][ids]).numpy().astype(float)
                if target_inverse:
                    prediction = np.maximum(np.expm1(prediction), 0)
                actual = d['actual'][ids].astype(float)
                assert np.isfinite(prediction).all()
                error = prediction - actual
                archive.update({f'{split}_row_id': d['row_id'][ids], f'{split}_actual': actual,
                                f'{split}_predicted': prediction, f'{split}_group_id': d['groups'][ids]})
                scores.append(dict(case=CASE, model=job['model'], seed=seed, split=split,
                    protocol=contract['split_policy'], scoring=contract['scoring'], config_digest=digest,
                    parameters=job['parameters'], epochs=len(history['train_loss']), seconds=elapsed,
                    rmse=float(np.sqrt(np.mean(error**2))), mae=float(np.mean(abs(error))),
                    r2=float(1 - np.sum(error**2) / np.sum((actual - actual.mean())**2))))
            state = dict(model_state=trainer.model.state_dict(), x_mean=trainer.x_scaler.mean,
                         x_std=trainer.x_scaler.std, y_mean=trainer.y_scaler.mean, y_std=trainer.y_scaler.std,
                         coord_center=trainer.coord_scaler.center, coord_scale=trainer.coord_scaler.scale,
                         context_ids=trainer.context_ids, context_x=trainer.context_x,
                         context_coordinates=trainer.context_coordinates, context_y=trainer.context_y)
            torch.save(dict(state=state, configuration=configuration, seed=seed),
                       directory / f'seed_{seed}_checkpoint.pt')
            np.savez_compressed(directory / f'seed_{seed}_predictions.npz', **archive)
            pd.DataFrame(history).to_csv(directory / f'seed_{seed}_history.csv', index=False)
            # Reload the checkpoint into a fresh model before declaring the seed complete.
            restored = build_trainer(job, DEVICE)
            saved = torch.load(directory / f'seed_{seed}_checkpoint.pt', map_location='cpu', weights_only=False)['state']
            restored.model.load_state_dict(saved['model_state'], strict=True)
            for name in ('context_ids', 'context_x', 'context_coordinates', 'context_y'):
                setattr(restored, name, saved[name])
            restored.x_scaler.mean, restored.x_scaler.std = saved['x_mean'], saved['x_std']
            restored.y_scaler.mean, restored.y_scaler.std = saved['y_mean'], saved['y_std']
            restored.coord_scaler.center, restored.coord_scaler.scale = saved['coord_center'], saved['coord_scale']
            audits = []
            for split in ('validation', 'test'):
                ids = d[f'{split}_ids']
                prediction = restored.predict(d['x'][ids], d['u'][ids]).numpy().astype(float)
                if target_inverse:
                    prediction = np.maximum(np.expm1(prediction), 0)
                np.testing.assert_allclose(prediction, archive[f'{split}_predicted'], rtol=1e-5, atol=1e-6)
                audits.append(dict(seed=seed, split=split, rows=len(ids), prediction_reload_pass=True))
            pd.DataFrame(audits).to_csv(directory / f'seed_{seed}_audit.csv', index=False)
            pd.DataFrame(scores).to_csv(metric_path, index=False)
            del trainer, restored, state, saved
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

In [8]:
model_records = []
for job in model_jobs:
    d = np.load(DATA_DIR / job['data'])
    source = SCRIPTS / job['reference_dir'] if MODE == 'replay' else OUTPUT_DIR / job['id']
    map_u = np.load(DATA_DIR / 'map_coordinates.npz')['coordinates']
    for seed in SEEDS:
        if MODE == 'replay':
            for suffix in ('metrics.csv', 'predictions.npz', 'checkpoint.pt', 'audit.csv', 'history.csv'):
                name = f'seed_{seed}_{suffix}'
                assert hashlib.sha256((source / name).read_bytes()).hexdigest() == job['files'][name]
        frame = pd.read_csv(source / f'seed_{seed}_metrics.csv')
        prediction = np.load(source / f'seed_{seed}_predictions.npz')
        audit = pd.read_csv(source / f'seed_{seed}_audit.csv')
        assert len(frame) == 2 and audit.prediction_reload_pass.all()
        checkpoint = torch.load(source / f'seed_{seed}_checkpoint.pt', map_location='cpu', weights_only=False)
        state = checkpoint['state']
        assert set(state['context_ids'].tolist()) == set(d['train_ids'].tolist())
        for split in ('validation', 'test'):
            ids = d[f'{split}_ids']
            np.testing.assert_array_equal(prediction[f'{split}_row_id'], d['row_id'][ids])
            actual = prediction[f'{split}_actual'].astype(float)
            predicted = prediction[f'{split}_predicted'].astype(float)
            np.testing.assert_allclose(actual, d['actual'][ids], rtol=1e-6, atol=1e-6)
            error = predicted - actual
            values = dict(rmse=np.sqrt(np.mean(error**2)), mae=np.mean(abs(error)),
                          r2=1 - np.sum(error**2) / np.sum((actual - actual.mean())**2))
            row = frame[frame.split == split].iloc[0].to_dict()
            for key, value in values.items():
                np.testing.assert_allclose(value, row[key], rtol=2e-5, atol=1e-6)
            row.update(values, case=CASE, model_label=job['label'], job_id=job['id'])
            if SHORT == 'SIC':
                tail = actual >= np.quantile(actual, .99)
                row['upper_1pct_sse_share'] = np.sum(error[tail]**2) / np.sum(error**2)
                row['body_rmse'] = np.sqrt(np.mean(error[~tail]**2))
                row['log1p_rmse'] = np.sqrt(np.mean((np.log1p(predicted)-np.log1p(actual))**2))
            if SHORT == 'SHP':
                details = pd.DataFrame(dict(group=d['groups'][ids], row_id=d['row_id'][ids],
                                            actual=actual, predicted=predicted, squared=error**2))
                row['equal_group_rmse'] = np.sqrt(details.groupby('group').squared.mean().mean())
                for group, g in details.groupby('group'):
                    index = np.flatnonzero(d['row_id'] == g.row_id.iloc[0])[0]
                    case_groups.append(dict(case=CASE, model=job['label'], job_id=job['id'], seed=seed,
                        split=split, group=group, n=len(g), rmse=np.sqrt(g.squared.mean()),
                        mean_residual=(g.predicted-g.actual).mean(), x=map_u[index,0], y=map_u[index,1]))
            case_metrics.append(row)
            model_records.append(row)
            if split == 'test':
                case_predictions.append(pd.DataFrame(dict(case=CASE, model=job['label'], job_id=job['id'],
                    seed=seed, row_id=d['row_id'][ids], actual=actual, predicted=predicted,
                    residual=error, group=d['groups'][ids], x=map_u[ids,0], y=map_u[ids,1])))
        if job['model'] == 'dlan':
            names = {'CE':['Space x','Space y','Time'], 'SIC':['Origin x','Origin y','Destination x','Destination y'],
                     'SHP':['Space x','Space y'], 'GVI':['Space x','Space y','Time']}[SHORT]
            for key, value in state['model_state'].items():
                if key.endswith('raw_bandwidth'):
                    bandwidth = (torch.nn.functional.softplus(value) + 1e-6).numpy()
                    for axis in range(bandwidth.shape[0]):
                        case_bandwidths.append(dict(case=CASE, job_id=job['id'], seed=seed,
                                                    axis=names[axis], bandwidth=float(bandwidth[axis,0])))
        if SHORT == 'GVI' and job['model'] == 'gcnnwr':
            ids = d['test_ids']
            sx = (torch.as_tensor(d['x'][ids], dtype=torch.float32)-state['x_mean']) / state['x_std']
            design = torch.cat((torch.ones_like(sx[:,:1]), sx), 1)
            ols = (design @ state['model_state']['network.ols_coefficients'] * state['y_std'] + state['y_mean']).numpy()
            actual = d['actual'][ids]
            case_ols.append(dict(case=CASE, seed=seed, rmse=np.sqrt(np.mean((ols-actual)**2))))
            if seed == 1:
                case_predictions.append(pd.DataFrame(dict(case=CASE, model='OLS buffer', job_id=job['id'],
                    seed=seed, row_id=d['row_id'][ids], actual=actual, predicted=ols,
                    residual=ols-actual, group=d['groups'][ids], x=map_u[ids,0], y=map_u[ids,1])))
pd.DataFrame(model_records).query("split == 'test'").groupby('job_id')[['rmse','mae','r2']].agg(['mean','std'])

rmse                    mae             \
                                   mean         std       mean        std   
job_id                                                                      
geoaggregator_archived_main  662.084584  348.936702  56.346975  13.748825   
geoaggregator_main_capacity  772.733981  835.035352  60.187094  24.683249   

                                   r2            
                                 mean       std  
job_id                                           
geoaggregator_archived_main  0.847527  0.175877  
geoaggregator_main_capacity  0.659212  0.827648

### GNNWR

In [9]:
model_jobs = [j for j in CASE_JOBS if j['model'] in ('gnnwr','gtnnwr')]
pd.DataFrame([{'job':j['id'], 'parameters':j['parameters'], 'candidates':j['candidates'],
               'roles':', '.join(j['roles'])} for j in model_jobs])

,job,parameters,candidates,roles
0,gnnwr_main_capacity,7069,256,"main, capacity"
1,gnnwr_archived_main,6957,256,archived_main


In [10]:
# Training is explicit. The default replay mode never enters this block.
if MODE == 'train':
    for job in model_jobs:
        d = np.load(DATA_DIR / job['data'])
        source = SCRIPTS / job['reference_dir']
        original = json.loads((source / 'configuration.json').read_text())
        target_inverse = original['contract'].get('target_inverse')
        directory = OUTPUT_DIR / job['id']
        directory.mkdir(exist_ok=True)
        configuration = {'specification': job, 'mode': MODE, 'manifest_digest': manifest_digest,
                         'training': {**job['training'], 'device': DEVICE},
                         'data_sha256': hashlib.sha256((DATA_DIR / job['data']).read_bytes()).hexdigest(),
                         'model_hashes': {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                                          for p in sorted((SCRIPTS / 'models').glob('*.py'))},
                         'python': platform.python_version(), 'torch': torch.__version__}
        digest = hashlib.sha256(json.dumps(configuration, sort_keys=True).encode()).hexdigest()
        configuration['config_digest'] = digest
        config_path = directory / 'configuration.json'
        if config_path.exists():
            assert json.loads(config_path.read_text())['config_digest'] == digest, 'Choose a new RUN_ID'
        else:
            config_path.write_text(json.dumps(configuration, indent=2))
        for seed in SEEDS:
            metric_path = directory / f'seed_{seed}_metrics.csv'
            if metric_path.exists():
                assert set(pd.read_csv(metric_path).config_digest) == {digest}
                assert all((directory / f'seed_{seed}_{s}').is_file()
                           for s in ('checkpoint.pt', 'predictions.npz', 'history.csv', 'audit.csv'))
                continue
            random.seed(seed)
            np.random.seed(seed)
            torch.manual_seed(seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(seed)
            trainer = build_trainer(job, DEVICE)
            started = time.perf_counter()
            history = trainer.fit(d['x'], d['u'], d['y'], d['train_ids'], d['validation_ids'], verbose=False)
            elapsed = time.perf_counter() - started
            archive, scores = {}, []
            for split in ('validation', 'test'):
                ids = d[f'{split}_ids']
                prediction = trainer.predict(d['x'][ids], d['u'][ids]).numpy().astype(float)
                if target_inverse:
                    prediction = np.maximum(np.expm1(prediction), 0)
                actual = d['actual'][ids].astype(float)
                assert np.isfinite(prediction).all()
                error = prediction - actual
                archive.update({f'{split}_row_id': d['row_id'][ids], f'{split}_actual': actual,
                                f'{split}_predicted': prediction, f'{split}_group_id': d['groups'][ids]})
                scores.append(dict(case=CASE, model=job['model'], seed=seed, split=split,
                    protocol=contract['split_policy'], scoring=contract['scoring'], config_digest=digest,
                    parameters=job['parameters'], epochs=len(history['train_loss']), seconds=elapsed,
                    rmse=float(np.sqrt(np.mean(error**2))), mae=float(np.mean(abs(error))),
                    r2=float(1 - np.sum(error**2) / np.sum((actual - actual.mean())**2))))
            state = dict(model_state=trainer.model.state_dict(), x_mean=trainer.x_scaler.mean,
                         x_std=trainer.x_scaler.std, y_mean=trainer.y_scaler.mean, y_std=trainer.y_scaler.std,
                         coord_center=trainer.coord_scaler.center, coord_scale=trainer.coord_scaler.scale,
                         context_ids=trainer.context_ids, context_x=trainer.context_x,
                         context_coordinates=trainer.context_coordinates, context_y=trainer.context_y)
            torch.save(dict(state=state, configuration=configuration, seed=seed),
                       directory / f'seed_{seed}_checkpoint.pt')
            np.savez_compressed(directory / f'seed_{seed}_predictions.npz', **archive)
            pd.DataFrame(history).to_csv(directory / f'seed_{seed}_history.csv', index=False)
            # Reload the checkpoint into a fresh model before declaring the seed complete.
            restored = build_trainer(job, DEVICE)
            saved = torch.load(directory / f'seed_{seed}_checkpoint.pt', map_location='cpu', weights_only=False)['state']
            restored.model.load_state_dict(saved['model_state'], strict=True)
            for name in ('context_ids', 'context_x', 'context_coordinates', 'context_y'):
                setattr(restored, name, saved[name])
            restored.x_scaler.mean, restored.x_scaler.std = saved['x_mean'], saved['x_std']
            restored.y_scaler.mean, restored.y_scaler.std = saved['y_mean'], saved['y_std']
            restored.coord_scaler.center, restored.coord_scaler.scale = saved['coord_center'], saved['coord_scale']
            audits = []
            for split in ('validation', 'test'):
                ids = d[f'{split}_ids']
                prediction = restored.predict(d['x'][ids], d['u'][ids]).numpy().astype(float)
                if target_inverse:
                    prediction = np.maximum(np.expm1(prediction), 0)
                np.testing.assert_allclose(prediction, archive[f'{split}_predicted'], rtol=1e-5, atol=1e-6)
                audits.append(dict(seed=seed, split=split, rows=len(ids), prediction_reload_pass=True))
            pd.DataFrame(audits).to_csv(directory / f'seed_{seed}_audit.csv', index=False)
            pd.DataFrame(scores).to_csv(metric_path, index=False)
            del trainer, restored, state, saved
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

In [11]:
model_records = []
for job in model_jobs:
    d = np.load(DATA_DIR / job['data'])
    source = SCRIPTS / job['reference_dir'] if MODE == 'replay' else OUTPUT_DIR / job['id']
    map_u = np.load(DATA_DIR / 'map_coordinates.npz')['coordinates']
    for seed in SEEDS:
        if MODE == 'replay':
            for suffix in ('metrics.csv', 'predictions.npz', 'checkpoint.pt', 'audit.csv', 'history.csv'):
                name = f'seed_{seed}_{suffix}'
                assert hashlib.sha256((source / name).read_bytes()).hexdigest() == job['files'][name]
        frame = pd.read_csv(source / f'seed_{seed}_metrics.csv')
        prediction = np.load(source / f'seed_{seed}_predictions.npz')
        audit = pd.read_csv(source / f'seed_{seed}_audit.csv')
        assert len(frame) == 2 and audit.prediction_reload_pass.all()
        checkpoint = torch.load(source / f'seed_{seed}_checkpoint.pt', map_location='cpu', weights_only=False)
        state = checkpoint['state']
        assert set(state['context_ids'].tolist()) == set(d['train_ids'].tolist())
        for split in ('validation', 'test'):
            ids = d[f'{split}_ids']
            np.testing.assert_array_equal(prediction[f'{split}_row_id'], d['row_id'][ids])
            actual = prediction[f'{split}_actual'].astype(float)
            predicted = prediction[f'{split}_predicted'].astype(float)
            np.testing.assert_allclose(actual, d['actual'][ids], rtol=1e-6, atol=1e-6)
            error = predicted - actual
            values = dict(rmse=np.sqrt(np.mean(error**2)), mae=np.mean(abs(error)),
                          r2=1 - np.sum(error**2) / np.sum((actual - actual.mean())**2))
            row = frame[frame.split == split].iloc[0].to_dict()
            for key, value in values.items():
                np.testing.assert_allclose(value, row[key], rtol=2e-5, atol=1e-6)
            row.update(values, case=CASE, model_label=job['label'], job_id=job['id'])
            if SHORT == 'SIC':
                tail = actual >= np.quantile(actual, .99)
                row['upper_1pct_sse_share'] = np.sum(error[tail]**2) / np.sum(error**2)
                row['body_rmse'] = np.sqrt(np.mean(error[~tail]**2))
                row['log1p_rmse'] = np.sqrt(np.mean((np.log1p(predicted)-np.log1p(actual))**2))
            if SHORT == 'SHP':
                details = pd.DataFrame(dict(group=d['groups'][ids], row_id=d['row_id'][ids],
                                            actual=actual, predicted=predicted, squared=error**2))
                row['equal_group_rmse'] = np.sqrt(details.groupby('group').squared.mean().mean())
                for group, g in details.groupby('group'):
                    index = np.flatnonzero(d['row_id'] == g.row_id.iloc[0])[0]
                    case_groups.append(dict(case=CASE, model=job['label'], job_id=job['id'], seed=seed,
                        split=split, group=group, n=len(g), rmse=np.sqrt(g.squared.mean()),
                        mean_residual=(g.predicted-g.actual).mean(), x=map_u[index,0], y=map_u[index,1]))
            case_metrics.append(row)
            model_records.append(row)
            if split == 'test':
                case_predictions.append(pd.DataFrame(dict(case=CASE, model=job['label'], job_id=job['id'],
                    seed=seed, row_id=d['row_id'][ids], actual=actual, predicted=predicted,
                    residual=error, group=d['groups'][ids], x=map_u[ids,0], y=map_u[ids,1])))
        if job['model'] == 'dlan':
            names = {'CE':['Space x','Space y','Time'], 'SIC':['Origin x','Origin y','Destination x','Destination y'],
                     'SHP':['Space x','Space y'], 'GVI':['Space x','Space y','Time']}[SHORT]
            for key, value in state['model_state'].items():
                if key.endswith('raw_bandwidth'):
                    bandwidth = (torch.nn.functional.softplus(value) + 1e-6).numpy()
                    for axis in range(bandwidth.shape[0]):
                        case_bandwidths.append(dict(case=CASE, job_id=job['id'], seed=seed,
                                                    axis=names[axis], bandwidth=float(bandwidth[axis,0])))
        if SHORT == 'GVI' and job['model'] == 'gcnnwr':
            ids = d['test_ids']
            sx = (torch.as_tensor(d['x'][ids], dtype=torch.float32)-state['x_mean']) / state['x_std']
            design = torch.cat((torch.ones_like(sx[:,:1]), sx), 1)
            ols = (design @ state['model_state']['network.ols_coefficients'] * state['y_std'] + state['y_mean']).numpy()
            actual = d['actual'][ids]
            case_ols.append(dict(case=CASE, seed=seed, rmse=np.sqrt(np.mean((ols-actual)**2))))
            if seed == 1:
                case_predictions.append(pd.DataFrame(dict(case=CASE, model='OLS buffer', job_id=job['id'],
                    seed=seed, row_id=d['row_id'][ids], actual=actual, predicted=ols,
                    residual=ols-actual, group=d['groups'][ids], x=map_u[ids,0], y=map_u[ids,1])))
pd.DataFrame(model_records).query("split == 'test'").groupby('job_id')[['rmse','mae','r2']].agg(['mean','std'])

rmse                     mae                   r2  \
                            mean         std        mean        std      mean   
job_id                                                                          
gnnwr_archived_main  3254.053951  740.916956  207.167132  38.666208 -2.084026   
gnnwr_main_capacity  3568.803945  969.208499  210.937533  45.807368 -2.779379   

                               
                          std  
job_id                         
gnnwr_archived_main  1.385959  
gnnwr_main_capacity  2.043676

### GCNNWR

Not run: the implemented GCNNWR requires two-dimensional planar coordinates. Time or endpoint dimensions are retained, so no reduced-coordinate substitute is fitted.

## Results summary

In [12]:
metrics = pd.DataFrame(case_metrics)
metrics.to_csv(OUTPUT_DIR / 'all_metrics.csv', index=False)
pd.concat(case_predictions, ignore_index=True).to_csv(OUTPUT_DIR / 'all_predictions.csv', index=False)
pd.DataFrame(case_bandwidths, columns=['case','job_id','seed','axis','bandwidth']).to_csv(OUTPUT_DIR / 'bandwidths.csv', index=False)
pd.DataFrame(case_groups, columns=['case','model','job_id','seed','split','group','n','rmse','mean_residual','x','y']).to_csv(OUTPUT_DIR / 'groups.csv', index=False)
pd.DataFrame(case_ols, columns=['case','seed','rmse']).to_csv(OUTPUT_DIR / 'ols.csv', index=False)
(OUTPUT_DIR / 'completion.json').write_text(json.dumps(dict(case=CASE, mode=MODE,
    manifest_digest=manifest_digest, seeds=SEEDS, jobs=[j['id'] for j in CASE_JOBS],
    prediction_sets_checked=len(metrics)), indent=2))
main_ids = [j['id'] for j in CASE_JOBS if 'main' in j['roles']]
metrics[(metrics.job_id.isin(main_ids)) & (metrics.split=='test')].groupby('model_label').agg(
    runs=('seed','nunique'), parameters=('parameters','first'), rmse_mean=('rmse','mean'),
    rmse_sd=('rmse','std'), mae_mean=('mae','mean'), r2_mean=('r2','mean'))

,runs,parameters,rmse_mean,rmse_sd,mae_mean,r2_mean
model_label,,,,,,
DLAN,10,7069,565.276291,154.306900,50.281942,0.905120
GNNWR,10,7069,3568.803945,969.208499,210.937533,-2.779379
GeoAggregator,10,7069,772.733981,835.035352,60.187094,0.659212


In [13]:
# The last completed case assembles the complete figure-input bundle.
run_dir = SCRIPTS / 'outputs' / RUN_ID
pending = [c for c in manifest['cases'] if not (run_dir / c / 'completion.json').is_file()]
staging = run_dir / 'evidence'
if not pending:
    for c in manifest['cases']:
        finished = json.loads((run_dir / c / 'completion.json').read_text())
        assert finished['mode'] == MODE and finished['manifest_digest'] == manifest_digest
        assert finished['seeds'] == SEEDS
    staging.mkdir(exist_ok=True)
    all_metrics = pd.concat([pd.read_csv(run_dir / c / 'all_metrics.csv') for c in manifest['cases']], ignore_index=True)
    all_predictions = pd.concat([pd.read_csv(run_dir / c / 'all_predictions.csv') for c in manifest['cases']], ignore_index=True)
    all_bands = pd.concat([pd.read_csv(run_dir / c / 'bandwidths.csv') for c in manifest['cases']], ignore_index=True)
    main_keys = {(j['case'],j['id']) for j in manifest['jobs'] if 'main' in j['roles']}
    archive_keys = {(j['case'],j['id']) for j in manifest['jobs']
                    if ('main' in j['roles'] and j['short']!='SIC') or 'archived_main' in j['roles']}
    for prefix, keys in [('reported',main_keys), ('matched',archive_keys)]:
        selected = all_metrics[[ (r.case,r.job_id) in keys for r in all_metrics.itertuples() ]].copy()
        for column in ('equal_group_rmse','upper_1pct_sse_share','body_rmse'):
            if column not in selected: selected[column] = np.nan
        summary = selected.groupby(['case','model_label','split']).agg(
            runs=('seed','nunique'), parameters=('parameters','first'), rmse_mean=('rmse','mean'),
            rmse_sd=('rmse','std'), rmse_median=('rmse','median'), mae_mean=('mae','mean'),
            r2_mean=('r2','mean'), seconds_mean=('seconds','mean'), seconds_sd=('seconds','std'),
            epochs_mean=('epochs','mean'), equal_group_rmse=('equal_group_rmse','mean')).reset_index()
        predictions = all_predictions[[ (r.case,r.job_id) in keys for r in all_predictions.itertuples() ]].copy()
        bandwidths = all_bands[[ (r.case,r.job_id) in keys for r in all_bands.itertuples() ]].copy()
        selected.to_csv(staging / f'{prefix}_metrics.csv', index=False)
        summary.to_csv(staging / f'{prefix}_summary.csv', index=False)
        predictions.to_csv(staging / f'{prefix}_predictions.csv', index=False)
        bandwidths.to_csv(staging / f'{prefix}_bandwidths.csv', index=False)
    pd.read_csv(run_dir / 'SHP/groups.csv').to_csv(staging / 'matched_hp_groups.csv', index=False)
    pd.read_csv(run_dir / 'GVI/ols.csv').to_csv(staging / 'gvi_ols_buffer.csv', index=False)
    supplement = []
    supplementary_predictions = []
    for job in manifest['jobs']:
        if 'supplement_experiment' not in job: continue
        frame = all_metrics[(all_metrics.case==job['case']) & (all_metrics.job_id==job['id'])].copy()
        frame['experiment'] = job['supplement_experiment']
        # Preserve the existing supplementary metric label (predictions use GVI).
        if job['supplement_experiment'] == 'GVI 3D': frame['case'] = 'GVI-ST'
        frame['model_label'] = job['supplement_label']
        supplement.append(frame)
        pred = all_predictions[(all_predictions.case==job['case']) & (all_predictions.job_id==job['id'])].copy()
        pred['experiment'] = job['supplement_experiment']
        pred['model'] = job['supplement_label']
        supplementary_predictions.append(pred)
    supplement = pd.concat(supplement, ignore_index=True)
    supplement.to_csv(staging / 'supplement_metrics.csv', index=False)
    pd.concat(supplementary_predictions, ignore_index=True).to_csv(staging / 'supplement_predictions.csv', index=False)
    supplement.groupby(['experiment','case','model_label','split']).agg(
        runs=('seed','nunique'), parameters=('parameters','first'), rmse_mean=('rmse','mean'),
        rmse_sd=('rmse','std'), rmse_median=('rmse','median'), rmse_worst=('rmse','max'),
        mae_mean=('mae','mean'), r2_mean=('r2','mean'), seconds_mean=('seconds','mean'),
        epochs_mean=('epochs','mean')).reset_index().to_csv(staging / 'supplement_summary.csv', index=False)
    factorial = []
    for job in manifest['jobs']:
        if 'factorial' not in job['roles']: continue
        frame = all_metrics[(all_metrics.case==job['case']) & (all_metrics.job_id==job['id'])].copy()
        frame['width'], frame['candidates'] = job['factorial_width'], job['candidates']
        factorial.append(frame)
    factorial = pd.concat(factorial, ignore_index=True)
    factorial.to_csv(staging / 'sic_factorial_metrics.csv', index=False)
    factorial.query("split == 'test'").groupby(['width','candidates']).agg(
        runs=('seed','nunique'), parameters=('parameters','first'), rmse_mean=('rmse','mean'),
        rmse_sd=('rmse','std'), mae_mean=('mae','mean'), rmse_worst=('rmse','max'),
        seconds_mean=('seconds','mean')).reset_index().to_csv(staging / 'sic_factorial_summary.csv', index=False)
    shutil.copy2(SCRIPTS / 'data/historical/gaussian_ablation.csv', staging / 'gaussian_ablation.csv')
    (staging / 'reproduction_manifest.json').write_text(json.dumps(dict(mode=MODE, run_id=RUN_ID,
        manifest_digest=manifest_digest, jobs=len(manifest['jobs']), seeds=SEEDS), indent=2))
{'pending_cases':pending, 'evidence_directory':str(staging), 'published':False}

{'pending_cases': ['SHP', 'GVI'],
 'evidence_directory': '/home/yigong/Works/SNWA/article/scripts/outputs/replay/evidence',
 'published': False}

In [14]:
# Explicit opt-in only. This cell never executes the plotting notebook.
if PUBLISH_EVIDENCE:
    assert not pending, 'Complete all four case notebooks first'
    article_dir = SCRIPTS.parent
    destination = article_dir / 'evidence'
    destination.mkdir(exist_ok=True)
    backup = run_dir / 'evidence_before_publish'
    backup.mkdir(exist_ok=True)
    for source in staging.iterdir():
        target = destination / source.name
        if target.exists() and not (backup / source.name).exists():
            shutil.copy2(target, backup / source.name)
        shutil.copy2(source, target)
{'published': bool(PUBLISH_EVIDENCE), 'plot_notebook_executed': False}

{'published': False, 'plot_notebook_executed': False}